# ATOM Training Notebook

Este notebook configura o ambiente `Atom-v1`, treina PPO usando o modelo local e salva checkpoints, logs e vídeos.

In [ ]:
import os
import sys

import gymnasium as gym
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# Instala dependências Python necessárias (opcional). Execute manualmente em sistemas locais.
import importlib, subprocess, sys
pkgs = ['pyvirtualdisplay','imageio[ffmpeg]','stable-baselines3','gymnasium','mujoco']
for pkg in pkgs:
    name = pkg.split('[')[0]
    try:
        importlib.import_module(name)
    except Exception:
        print(f'Instalando {pkg} via pip...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg])
print('Pacotes pip instalados (ou já presentes).')


In [ ]:
# Configura display virtual para MuJoCo
import os
from pyvirtualdisplay import Display

NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
    with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
        f.write('''{
    "file_format_version" : "1.0.0",
    "ICD" : {
        "library_path" : "libEGL_nvidia.so.0"
    }
}
''')

os.environ['MUJOCO_GL'] = 'egl'
virtual_display = Display(visible=0, size=(1920, 1080))
virtual_display.start()
print('Virtual display started:', virtual_display)


In [ ]:
# Define o passo atual de treinamento e configura diretórios
current_step = 'atom_step_001'

base_dir = os.path.join(os.getcwd(), 'saida_treino_atom')
work_dir = os.path.join(base_dir, current_step)
log_dir = os.path.join(work_dir, 'logs')
best_dir = os.path.join(work_dir, 'best')
video_dir = os.path.join(work_dir, 'video')
tf_log_dir = os.path.join(log_dir, 'tensorboard')

for path in [work_dir, log_dir, best_dir, video_dir, tf_log_dir]:
    os.makedirs(path, exist_ok=True)

print('Work dir:', work_dir)
print('Log dir:', log_dir)
print('Best model dir:', best_dir)
print('Video dir:', video_dir)
print('Tensorboard dir:', tf_log_dir)


In [ ]:
# Usa o repositório local op3_model e garante que o ambiente Atom esteja registrado
atom_repo = os.path.join(os.getcwd(), 'op3_model')

if not os.path.isdir(atom_repo):
    raise FileNotFoundError(f'Não foi encontrado o repositório local op3_model em {atom_repo}')

atom_src = os.path.join(atom_repo, 'src')
if atom_src not in sys.path:
    sys.path.insert(0, atom_src)

# Instalação local opcional, útil se ainda não estiver no ambiente
try:
    import atom  # noqa: F401
except ImportError:
    os.system(f'pip install -e {atom_repo}')
    import atom  # noqa: F401

print('Atom env registered from:', atom.__file__)


In [ ]:
# Ambiente Atom já registrado na célula anterior; nenhum passo adicional de importação necessário.


In [ ]:
# Define hiperparâmetros de treino para o Atom
algo = 'ppo'
lr = 3e-4
device = 'cpu'

n_timesteps = 5_000_000
save_freq = min(100_000, int(n_timesteps / 10))
eval_freq = min(200_000, int(n_timesteps / 10))
max_episode_steps = 1000

env_kwargs = {
    'keep_alive_weight': 1.0,
    'control_weight': 1e-3,
    'target_distance': 3.0,
    'velocity_weight': 3.0,
    'reach_target_reward': 10.0,
    'knee_flex_weight': 1e-3,
    'feet_up_weight': 1e-3,
    'feet_misalign_weight': 0.05,
    'max_timestep': max_episode_steps,
}

print('Training hyperparameters configured')
print('Algorithm:', algo)
print('Environment kwargs:', env_kwargs)


In [ ]:
# Executa o treinamento do Atom usando PPO local
print('=== Treinando PPO para Atom-v1 ===')

train_env = gym.make('Atom-v1', render_mode=None, **env_kwargs)
train_env = Monitor(train_env, filename=os.path.join(log_dir, 'train'))

eval_env = gym.make('Atom-v1', render_mode=None, **env_kwargs)
eval_env = Monitor(eval_env, filename=os.path.join(log_dir, 'eval'))

model = PPO(
    'MlpPolicy',
    train_env,
    verbose=1,
    device=device,
    learning_rate=lr,
    n_steps=1024,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.0,
    tensorboard_log=tf_log_dir,
)

eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=best_dir,
    log_path=log_dir,
    eval_freq=eval_freq,
    n_eval_episodes=5,
    deterministic=True,
    render=False,
)

model.learn(
    total_timesteps=n_timesteps,
    callback=eval_callback,
    tb_log_name='ppo_atom',
)

final_model_path = os.path.join(best_dir, 'ppo_atom_final')
model.save(final_model_path)

print(f'Modelo final salvo em: {final_model_path}.zip')
print(f'Melhor modelo salvo em: {os.path.join(best_dir, "best_model.zip") }')

train_env.close()
eval_env.close()


In [ ]:
# Se quiser visualizar o modelo treinado localmente, use o script run_atom.py ou uma célula dedicada de renderização.
# O treinamento aqui salva best_model e ppo_atom_final no diretório saida_treino_atom/atom_step_001/best.

print('Use run_atom.py ou run_all_models_viewer.py para visualizar o modelo salvo.')


In [ ]:
# Verifica caminhos gerados e lista arquivos de saída
print('Diretórios finais:')
print('work_dir:', work_dir)
print('log_dir:', log_dir)
print('tf_log_dir:', tf_log_dir)
print('best_dir:', best_dir)
print('video_dir:', video_dir)

for directory in [log_dir, best_dir, video_dir]:
    print(f'\nConteúdo de {directory}:')
    for root, dirs, files in os.walk(directory):
        for name in files:
            print(os.path.join(root, name))
